# Advanced Analysis Techniques: Sensitivity & Uncertainty

This notebook demonstrates **advanced analysis techniques** for understanding parameter sensitivities and quantifying uncertainties in gas swelling model predictions.

## Learning Objectives

By the end of this tutorial, you will:
- Understand sensitivity analysis and why it matters
- Learn three sensitivity analysis methods (OAT, Morris, Sobol)
- Perform uncertainty quantification using Monte Carlo sampling
- Identify which parameters most affect model predictions
- Quantify prediction confidence intervals
- Create publication-quality sensitivity visualizations

## Why Sensitivity & Uncertainty Analysis?

**Sensitivity analysis** answers:
- Which parameters have the greatest influence on swelling?
- How sensitive is the model to parameter uncertainties?
- Are there interactions between parameters?
- Which parameters need accurate measurement?

**Uncertainty quantification** answers:
- What is the confidence interval for swelling predictions?
- How do parameter uncertainties propagate through the model?
- Which uncertainty sources contribute most to prediction uncertainty?
- How can we reduce prediction uncertainty?

## Methods Covered

1. **One-at-a-Time (OAT)**: Local sensitivity, quick screening
2. **Morris Method**: Global screening, captures nonlinearities
3. **Sobol Analysis**: Variance-based, comprehensive interaction analysis
4. **Monte Carlo UQ**: Propagate parameter uncertainties to predictions

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters
from gas_swelling.analysis.sensitivity import (
    OATAnalyzer,
    MorrisAnalyzer,
    SobolAnalyzer,
    ParameterRange,
    create_default_parameter_ranges
)
import warnings
warnings.filterwarnings('ignore')

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to perform advanced sensitivity and uncertainty analysis.")

## 1. Understanding Sensitivity Analysis Concepts

Before diving into calculations, let's understand key concepts:

In [ ]:
print("="*70)
print("SENSITIVITY ANALYSIS CONCEPTS")
print("="*70)

print("\n📊 LOCAL vs GLOBAL SENSITIVITY:")
print("   • Local (OAT): Sensitivity at nominal parameter values")
print("     - Vary one parameter, keep others fixed")
print("     - Fast, simple, but misses interactions")
print("")
print("   • Global (Morris, Sobol): Sensitivity over full parameter space")
print("     - Vary all parameters simultaneously")
print("     - Captures nonlinearities and interactions")
print("     - More computationally intensive")

print("\n📊 SENSITIVITY METRICS:")
print("   • Elasticity (OAT): (%Δ output) / (%Δ input)")
print("     - Value of 2.0 means 1% parameter change → 2% output change")
print("")
print("   • μ* (Morris): Mean of absolute elementary effects")
print("     - Measures overall influence")
print("     - Higher values = more important parameter")
print("")
print("   • S1, ST (Sobol): Variance-based indices")
print("     - S1: Main effect (parameter acting alone)")
print("     - ST: Total effect (including interactions)")
print("     - ST - S1: Strength of parameter interactions")

print("\n📊 WHEN TO USE EACH METHOD:")
print("   • OAT: Quick initial screening, limited parameters")
print("   • Morris: Medium-sized studies, screening many parameters")
print("   • Sobol: Final validation, detailed interaction analysis")
print("   • Monte Carlo: Uncertainty quantification, confidence intervals")

print("\n📊 COMPUTATIONAL COST:")
print("   Method      | Simulations (for k parameters)")
print("   " + "-"*50)
print("   OAT         | k × n_variations (e.g., 20-100)")
print("   Morris      | r × (k + 1) where r = 10-50 trajectories")
print("   Sobol       | N × (2k + 2) where N = 100-1000")
print("   Monte Carlo | N samples where N = 100-10000")

print("="*70)

## 2. Setup: Define Parameter Ranges

For sensitivity analysis, we need to define plausible ranges for each parameter. These ranges represent uncertainty in parameter values.

In [ ]:
# Create default parameter ranges
# These represent plausible uncertainty ranges for each parameter
param_ranges = create_default_parameter_ranges()

print("="*70)
print("PARAMETER RANGES FOR SENSITIVITY ANALYSIS")
print("="*70)
print("\nThese ranges represent plausible uncertainties in parameter values.")
print("\n{:25s} {:>12s} {:>12s} {:>12s} {:>12s}".format(
    "Parameter", "Min", "Nominal", "Max", "Distribution"))
print("-"*70)

for pr in param_ranges:
    print("{:25s} {:>12.3g} {:>12.3g} {:>12.3g} {:>12s}".format(
        pr.name, pr.min_value, pr.nominal_value, pr.max_value, pr.distribution
    ))

print("\n📝 Notes:")
print("   • Nominal: Best estimate or literature value")
print("   • Min/Max: Plausible lower/upper bounds")
print("   • Distribution: How to sample within bounds")
print("     - uniform: Equal probability everywhere")
print("     - normal: Bell curve centered at nominal")
print("     - loguniform: Uniform in log space (for orders of magnitude)")

## 3. One-at-a-Time (OAT) Sensitivity Analysis

OAT analysis is the simplest form of sensitivity analysis. We vary one parameter at a time while keeping all others at their nominal values.

In [ ]:
# Initialize OAT analyzer
oat_analyzer = OATAnalyzer(
    parameter_ranges=param_ranges,
    output_names=['swelling'],
    sim_time=7200000.0,  # 83.3 days
    t_eval_points=100
)

print("OAT Analyzer Configuration:")
print(f"  Parameters: {oat_analyzer.get_n_parameters()}")
print(f"  Simulation time: {oat_analyzer.sim_time/86400:.2f} days")
print(f"  Output metric: swelling (%)")

# Define percentage variations to test
percent_variations = [-20, -10, 10, 20]

print(f"\nRunning OAT analysis with variations: {percent_variations}%")
print("(This may take 2-3 minutes...)\n")

# Run OAT analysis
oat_results = oat_analyzer.run_oat_analysis(
    percent_variations=percent_variations,
    verbose=True
)

In [ ]:
# Process and visualize OAT results
# Extract elasticities and rank parameters
ranking_data = []
for result in oat_results:
    elasticity = result.sensitivities['swelling']['elasticity']
    ranking_data.append({
        'parameter': result.parameter_name,
        'elasticity': elasticity,
        'abs_elasticity': abs(elasticity),
        'normalized': result.sensitivities['swelling']['normalized'],
        'std': result.sensitivities['swelling']['std'],
        'baseline_swelling': result.baseline_outputs['swelling']
    })

# Sort by absolute elasticity
ranking_data.sort(key=lambda x: x['abs_elasticity'], reverse=True)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('OAT Sensitivity Analysis Results', fontsize=14, fontweight='bold')

# Plot 1: Tornado plot (parameter importance)
ax_tornado = axes[0, 0]
params = [r['parameter'] for r in ranking_data]
elasticities = [r['elasticity'] for r in ranking_data]
y_pos = np.arange(len(params))

colors = ['coral' if e > 0 else 'steelblue' for e in elasticities]
ax_tornado.barh(y_pos, elasticities, color=colors, alpha=0.7, edgecolor='black')
ax_tornado.set_yticks(y_pos)
ax_tornado.set_yticklabels(params, fontsize=9)
ax_tornado.set_xlabel('Elasticity |ΔY/Y| / |ΔX/X|', fontsize=10)
ax_tornado.set_title('Parameter Importance (Tornado Plot)', fontweight='bold')
ax_tornado.grid(True, alpha=0.3, axis='x')
ax_tornado.axvline(x=0, color='black', linestyle='--', linewidth=1)

# Plot 2: Elasticity ranking
ax_rank = axes[0, 1]
abs_elasticities = [r['abs_elasticity'] for r in ranking_data]
ax_rank.barh(y_pos, abs_elasticities, color='darkgreen', alpha=0.7, edgecolor='black')
ax_rank.set_yticks(y_pos)
ax_rank.set_yticklabels(params, fontsize=9)
ax_rank.set_xlabel('|Elasticity|', fontsize=10)
ax_rank.set_title('Absolute Sensitivity Ranking', fontweight='bold')
ax_rank.grid(True, alpha=0.3, axis='x')

# Plot 3: Parameter variation plots (top 4)
for idx, result in enumerate(ranking_data[:4]):
    ax = axes[1 + idx // 2, idx % 2]
    
    # Get the OAT result for this parameter
    oat_result = [r for r in oat_results if r.parameter_name == result['parameter']][0]
    
    variations = np.array(oat_result.variations)
    outputs = oat_result.outputs['swelling']
    nominal = oat_result.nominal_value
    baseline = oat_result.baseline_outputs['swelling']
    
    pct_change_param = (variations - nominal) / nominal * 100
    pct_change_output = (outputs - baseline) / baseline * 100
    
    ax.plot(pct_change_param, pct_change_output, 'o-', 
            linewidth=2.5, markersize=10, color='darkred', markeredgecolor='black')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel(f"{result['parameter']} change (%)", fontsize=10)
    ax.set_ylabel('Swelling change (%)', fontsize=10)
    ax.set_title(result['parameter'], fontweight='bold', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print ranking
print("\n" + "="*70)
print("PARAMETER SENSITIVITY RANKING (OAT Analysis)")
print("="*70)
print("\n{:3s} {:20s} {:>12s} {:>12s} {:>12s}".format(
    "#", "Parameter", "Elasticity", "|Elasticity|", "Baseline %"
))
print("-"*70)

for i, r in enumerate(ranking_data, 1):
    print("{:3d} {:20s} {:>12.3f} {:>12.3f} {:>12.4f}".format(
        i, r['parameter'], r['elasticity'], r['abs_elasticity'], r['baseline_swelling']
    ))

### Interpreting OAT Results

**Key observations:**

1. **Elasticity magnitude**: Measures sensitivity strength
   - |Elasticity| > 1.0: High sensitivity
   - |Elasticity| 0.1-1.0: Moderate sensitivity  
   - |Elasticity| < 0.1: Low sensitivity

2. **Elasticity sign**: Shows direction of effect
   - Positive: Parameter increase → swelling increase
   - Negative: Parameter increase → swelling decrease

3. **Linearity**: Straight lines = linear response
   - Curved lines indicate nonlinear effects

**Important caveat:** OAT only provides **local** sensitivity near nominal values. It misses parameter interactions!

## 4. Morris Elementary Effects Method

The Morris method explores the entire parameter space by computing "elementary effects" along random trajectories. It captures nonlinearities and parameter interactions.

In [ ]:
# Use a subset of parameters for Morris (computationally manageable)
morris_params = param_ranges[:5]  # First 5 parameters

print("Running Morris analysis on first 5 parameters...")
print("Parameters:")
for pr in morris_params:
    print(f"  - {pr.name}")

# Initialize Morris analyzer
morris_analyzer = MorrisAnalyzer(
    parameter_ranges=morris_params,
    output_names=['swelling'],
    sim_time=7200000.0,
    t_eval_points=100,
    num_levels=10,  # Discretization levels
    delta=None  # Will use default
)

# Number of trajectories
n_trajectories = 25
n_simulations = n_trajectories * (morris_analyzer.get_n_parameters() + 1)

print(f"\nMorris configuration:")
print(f"  Discretization levels: {morris_analyzer.num_levels}")
print(f"  Delta (step size): {morris_analyzer.delta:.3f}")
print(f"  Number of trajectories: {n_trajectories}")
print(f"  Total simulations: {n_simulations}")
print(f"\nRunning analysis... (this may take 3-5 minutes)\n")

# Run Morris analysis
morris_results = morris_analyzer.run_morris_analysis(
    n_trajectories=n_trajectories,
    verbose=True,
    random_state=42
)

In [ ]:
# Visualize Morris results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Morris Elementary Effects Results', fontsize=14, fontweight='bold')

# Extract data
param_names = morris_results.parameter_names
mu_star = morris_results.mu_star
mu = morris_results.mu
sigma = morris_results.sigma

# Plot 1: μ* (mu_star) ranking
ax1 = axes[0]
y_pos = np.arange(len(param_names))
ax1.barh(y_pos, mu_star, color='steelblue', alpha=0.7, edgecolor='black')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(param_names, fontsize=10)
ax1.set_xlabel('μ* (mu_star)', fontsize=11)
ax1.set_title('Parameter Influence Ranking', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Plot 2: μ* vs σ (influence vs nonlinearity)
ax2 = axes[1]
colors = ['coral' if m > 0 else 'steelblue' for m in mu]
scatter = ax2.scatter(mu_star, sigma, c=colors, s=150, alpha=0.7, edgecolors='black', linewidths=1.5)

# Add parameter labels
for i, name in enumerate(param_names):
    ax2.annotate(name, (mu_star[i], sigma[i]),
                xytext=(8, 8), textcoords='offset points',
                fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax2.set_xlabel('μ* (mu_star) - Overall Influence', fontsize=11)
ax2.set_ylabel('σ (sigma) - Nonlinearity/Interactions', fontsize=11)
ax2.set_title('Morris Interaction Plot', fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='coral', alpha=0.7, label='Positive effect (μ > 0)'),
    Patch(facecolor='steelblue', alpha=0.7, label='Negative effect (μ < 0)')
]
ax2.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

# Print ranking
print("\n" + "="*70)
print("MORRIS ELEMENTARY EFFECTS RESULTS")
print("="*70)
print("\n{:3s} {:20s} {:>12s} {:>12s} {:>12s}".format(
    "#", "Parameter", "μ*", "μ", "σ"
))
print("-"*70)

ranking = morris_results.get_ranking('swelling')
for i, (name, mu_star_val) in enumerate(ranking, 1):
    idx = morris_results.parameter_names.index(name)
    mu_val = morris_results.mu[idx]
    sigma_val = morris_results.sigma[idx]
    
    print("{:3d} {:20s} {:>12.4f} {:>12.4f} {:>12.4f}".format(
        i, name, mu_star_val, mu_val, sigma_val
    ))

print("\n📝 Interpretation:")
print("   • μ* (mu_star): Overall influence (higher = more important)")
print("   • μ (mu): Mean effect direction (positive/negative)")
print("   • σ (sigma): Nonlinearity and parameter interactions")
print("     - High σ indicates nonlinear effects or strong interactions")
print("     - Low σ indicates approximately linear behavior")

### Interpreting Morris Results

**The Morris plot (μ* vs σ) reveals:**

1. **High μ*, High σ** (upper right):
   - Strong influence with nonlinearities/interactions
   - Most critical parameters requiring careful treatment

2. **High μ*, Low σ** (lower right):
   - Strong influence, approximately linear
   - Can be modeled with simple linear approximations

3. **Low μ*, Low σ** (lower left):
   - Weak influence, linear behavior
   - Can potentially be fixed at nominal values

4. **Low μ*, High σ** (upper left):
   - Weak influence on average but highly nonlinear
   - May be important in specific regions of parameter space

## 5. Sobol Variance-Based Sensitivity Analysis

Sobol analysis decomposes the output variance into contributions from each parameter. It's the most comprehensive but computationally expensive method.

In [ ]:
# Use even fewer parameters for Sobol (very computationally intensive)
sobol_params = param_ranges[:3]  # First 3 parameters

print("Running Sobol analysis on first 3 parameters...")
print("Parameters:")
for pr in sobol_params:
    print(f"  - {pr.name}")

# Initialize Sobol analyzer
sobol_analyzer = SobolAnalyzer(
    parameter_ranges=sobol_params,
    output_names=['swelling'],
    sim_time=7200000.0,
    t_eval_points=100
)

# Sample size (N)
# For production use, N = 500-1000 is recommended
# Using N=100 for this demonstration
n_samples = 100
n_simulations = n_samples * (2 + sobol_analyzer.get_n_parameters())

print(f"\nSobol configuration:")
print(f"  Sample size (N): {n_samples}")
print(f"  Total simulations: {n_simulations}")
print(f"\n⚠️  Note: Sobol analysis is computationally intensive!")
print(f"    For production use, consider N = 500-1000 for robust results.")
print(f"\nRunning analysis... (this may take 5-10 minutes)\n")

# Run Sobol analysis
sobol_results = sobol_analyzer.run_sobol_analysis(
    n_samples=n_samples,
    verbose=True,
    random_state=42
)

In [ ]:
# Visualize Sobol results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sobol Variance-Based Sensitivity Analysis', fontsize=14, fontweight='bold')

# Extract data
param_names = sobol_results.parameter_names
s1 = sobol_results.S1[:, 0]  # First-order indices
st = sobol_results.ST[:, 0]   # Total-order indices
interactions = st - s1

y_pos = np.arange(len(param_names))

# Plot 1: Stacked bar (S1 + Interactions = ST)
axes[0].barh(y_pos, s1, color='steelblue', alpha=0.8, 
            label='S1 (main effect)', edgecolor='black')
axes[0].barh(y_pos, interactions, left=s1, color='coral', alpha=0.8,
            label='Interactions', edgecolor='black')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(param_names, fontsize=11)
axes[0].set_xlabel('Variance Fraction', fontsize=11)
axes[0].set_title('First-Order (S1) vs Total-Order (ST)', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].set_xlim(0, 1)

# Plot 2: S1 vs ST scatter
axes[1].scatter(s1, st, s=150, alpha=0.7, c='darkgreen', 
               edgecolors='black', linewidths=1.5)
# Add diagonal line (S1 = ST means no interactions)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=2,
             label='S1 = ST (no interactions)')

# Add parameter labels
for i, name in enumerate(param_names):
    axes[1].annotate(name, (s1[i], st[i]),
                    xytext=(8, -8), textcoords='offset points',
                    fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

axes[1].set_xlabel('S1 (first-order)', fontsize=11)
axes[1].set_ylabel('ST (total-order)', fontsize=11)
axes[1].set_title('Interaction Plot (ST vs S1)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# Print ranking
print("\n" + "="*70)
print("SOBOL VARIANCE-BASED RESULTS")
print("="*70)
print("\n{:3s} {:20s} {:>12s} {:>12s} {:>12s}".format(
    "#", "Parameter", "ST (total)", "S1 (main)", "Interaction"
))
print("-"*70)

ranking = sobol_results.get_ranking('swelling', order='ST')
for i, (name, st_val) in enumerate(ranking, 1):
    idx = sobol_results.parameter_names.index(name)
    s1_val = sobol_results.S1[idx, 0]
    interaction_val = st_val - s1_val
    
    print("{:3d} {:20s} {:>12.4f} {:>12.4f} {:>12.4f}".format(
        i, name, st_val, s1_val, interaction_val
    ))

print("\n📝 Interpretation:")
print("   • S1 (first-order): Fraction of variance from parameter acting alone")
print("   • ST (total-order): Fraction including all interactions")
print("   • ST - S1: Measures interaction strength")
print("     - ST ≈ S1: Parameter acts mostly independently")
print("     - ST >> S1: Parameter mainly affects through interactions")

### Interpreting Sobol Results

**Key insights from Sobol analysis:**

1. **S1 values**: Main effects
   - High S1: Parameter drives output variance on its own
   - These are "primary drivers" of the model

2. **ST values**: Total effects
   - Includes main effect + all interaction effects
   - High ST: Parameter is important, even if S1 is low

3. **ST - S1**: Interaction strength
   - Small difference: Parameter acts independently
   - Large difference: Parameter important mainly through interactions

**Practical implications:**
- High S1, high ST: Primary driver (measure accurately!)
- Low S1, high ST: Secondary driver through interactions
- Low S1, low ST: Can potentially be fixed at nominal value

## 6. Comparison of Sensitivity Analysis Methods

Let's compare the results from all three methods.

In [ ]:
# Create comparison table
print("\n" + "="*90)
print("COMPARISON OF SENSITIVITY ANALYSIS METHODS")
print("="*90)

comparison = """
+----------------------+----------+-----------+-----------+--------------------------+
| Feature              | OAT      | Morris    | Sobol     | Recommendation           |
+----------------------+----------+-----------+-----------+--------------------------+
| Type                 | Local    | Global    | Global    |                          |
| Computational cost   | Low      | Medium    | High      | Start with OAT           |
| Captures             | Linear   | Nonlinear | Nonlinear | Use Morris for screening |
| interactions         | No       | Partially | Yes       | Use Sobol for validation |
| Output metrics       | Elasticity| μ*, μ, σ | S1, ST    |                          |
| Ease of use          | High     | Medium    | Medium    |                          |
| Best for             | Screening| Screening | Compreh.  |                          |
| Time (for 5 params)  | ~1 min   | ~3 min    | ~8 min    |                          |
+----------------------+----------+-----------+-----------+--------------------------+
"""
print(comparison)

# Recommended workflow
print("\n📋 RECOMMENDED WORKFLOW:")
print("="*90)
print("\n1. Start with OAT analysis")
print("   ✓ Quick screening of many parameters")
print("   ✓ Identify obviously important/unimportant parameters")
print("   ✓ Understand local sensitivities")
print("   ✓ ~1 minute for 5 parameters")

print("\n2. Follow up with Morris analysis")
print("   ✓ Explore nonlinear effects")
print("   ✓ Detect parameter interactions")
print("   ✓ Refine parameter ranking")
print("   ✓ ~3-5 minutes for 5 parameters")

print("\n3. Finalize with Sobol analysis (if needed)")
print("   ✓ Quantify variance contributions")
print("   ✓ Measure interaction strengths")
print("   ✓ Provide rigorous sensitivity metrics")
print("   ✓ ~10-15 minutes for 3 parameters")

print("\n4. Document findings")
print("   ✓ Create sensitivity rankings")
print("   ✓ Identify parameters needing accurate measurement")
print("   ✓ Guide experimental validation efforts")

## 7. Monte Carlo Uncertainty Quantification

Now let's perform uncertainty quantification by propagating parameter uncertainties through the model using Monte Carlo sampling.

In [ ]:
# Monte Carlo UQ function
def monte_carlo_uq(param_ranges, n_samples=200, sim_time_days=83.3, random_state=42):
    """
    Perform Monte Carlo uncertainty quantification.
    
    Parameters:
    -----------
    param_ranges : list of ParameterRange
        Parameter ranges defining uncertainties
    n_samples : int
        Number of Monte Carlo samples
    sim_time_days : float
        Simulation time in days
    random_state : int
        Random seed for reproducibility
    
    Returns:
    --------
    dict : Results including all samples and statistics
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    # Get base parameters
    base_params = create_default_parameters()
    sim_time_sec = sim_time_days * 24 * 3600
    t_eval = np.linspace(0, sim_time_sec, 100)
    
    # Storage for results
    swelling_samples = []
    parameter_samples = {pr.name: [] for pr in param_ranges}
    
    print(f"Running Monte Carlo UQ with {n_samples} samples...")
    print(f"Parameters varied: {[pr.name for pr in param_ranges]}")
    print("\nProgress:")
    
    for i in range(n_samples):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  [{i+1}/{n_samples}]...")
        
        # Sample parameter values
        params = base_params.copy()
        for pr in param_ranges:
            value = pr.sample(1)[0]
            params[pr.name] = value
            parameter_samples[pr.name].append(value)
        
        # Run simulation
        try:
            model = GasSwellingModel(params)
            result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
            
            # Calculate swelling
            Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
            Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
            swelling = (Vb + Vf) * 100
            
            swelling_samples.append(swelling[-1])
        except Exception as e:
            print(f"    Warning: Simulation failed for sample {i+1}: {e}")
            swelling_samples.append(np.nan)
    
    # Convert to arrays
    swelling_samples = np.array(swelling_samples)
    for key in parameter_samples:
        parameter_samples[key] = np.array(parameter_samples[key])
    
    # Calculate statistics
    valid_samples = ~np.isnan(swelling_samples)
    swelling_valid = swelling_samples[valid_samples]
    
    stats = {
        'mean': np.mean(swelling_valid),
        'std': np.std(swelling_valid),
        'median': np.median(swelling_valid),
        'q5': np.percentile(swelling_valid, 5),
        'q95': np.percentile(swelling_valid, 95),
        'min': np.min(swelling_valid),
        'max': np.max(swelling_valid),
        'n_valid': np.sum(valid_samples),
        'n_failed': np.sum(~valid_samples)
    }
    
    return {
        'swelling_samples': swelling_samples,
        'parameter_samples': parameter_samples,
        'stats': stats
    }

# Run Monte Carlo UQ
# Use a subset of parameters for demonstration
uq_params = param_ranges[:5]

uq_results = monte_carlo_uq(
    param_ranges=uq_params,
    n_samples=200,
    sim_time_days=83.3,
    random_state=42
)

In [ ]:
# Visualize Monte Carlo results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Monte Carlo Uncertainty Quantification', fontsize=14, fontweight='bold')

# Extract data
swelling_samples = uq_results['swelling_samples']
valid = ~np.isnan(swelling_samples)
swelling_valid = swelling_samples[valid]
stats = uq_results['stats']

# Plot 1: Histogram of swelling predictions
axes[0, 0].hist(swelling_valid, bins=30, color='steelblue', alpha=0.7, 
                 edgecolor='black', density=True)
axes[0, 0].axvline(stats['mean'], color='red', linewidth=2, 
                    linestyle='--', label=f'Mean: {stats["mean"]:.4f}%')
axes[0, 0].axvline(stats['median'], color='green', linewidth=2, 
                    linestyle='--', label=f'Median: {stats["median"]:.4f}%')
axes[0, 0].axvline(stats['q5'], color='orange', linewidth=1.5, 
                    linestyle=':', label=f'5th percentile: {stats["q5"]:.4f}%')
axes[0, 0].axvline(stats['q95'], color='orange', linewidth=1.5, 
                    linestyle=':', label=f'95th percentile: {stats["q95"]:.4f}%')
axes[0, 0].set_xlabel('Swelling (%)', fontsize=11)
axes[0, 0].set_ylabel('Probability Density', fontsize=11)
axes[0, 0].set_title('Distribution of Swelling Predictions', fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Box plot with confidence interval
bp = axes[0, 1].boxplot([swelling_valid], vert=True, widths=0.5,
                          patch_artist=True, labels=['Swelling'])
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][0].set_alpha(0.7)
axes[0, 1].set_ylabel('Swelling (%)', fontsize=11)
axes[0, 1].set_title('Confidence Interval', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')
# Add statistics text
textstr = f"Mean: {stats['mean']:.4f}%\n"
textstr += f"Std: {stats['std']:.4f}%\n"
textstr += f"90% CI: [{stats['q5']:.4f}, {stats['q95']:.4f}]%"
axes[0, 1].text(1.15, 0.5, textstr, transform=axes[0, 1].transAxes,
                fontsize=10, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: Parameter vs swelling scatter (for most influential parameter)
param_samples = uq_results['parameter_samples']
# Find most correlated parameter
correlations = {}
for param_name in param_samples:
    corr = np.corrcoef(param_samples[param_name][valid], swelling_valid)[0, 1]
    correlations[param_name] = abs(corr)

most_correlated = max(correlations, key=correlations.get)
axes[1, 0].scatter(param_samples[most_correlated][valid], swelling_valid, 
                   alpha=0.5, s=20, color='darkgreen')
axes[1, 0].set_xlabel(most_correlated, fontsize=11)
axes[1, 0].set_ylabel('Swelling (%)', fontsize=11)
axes[1, 0].set_title(f'Swelling vs {most_correlated}', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Cumulative distribution function
sorted_data = np.sort(swelling_valid)
cumprob = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
axes[1, 1].plot(sorted_data, cumprob, linewidth=2, color='purple')
axes[1, 1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].axvline(stats['median'], color='gray', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(sorted_data, 0, cumprob, where=(sorted_data <= stats['q95']) & (sorted_data >= stats['q5']),
                         alpha=0.3, color='orange', label='90% CI')
axes[1, 1].set_xlabel('Swelling (%)', fontsize=11)
axes[1, 1].set_ylabel('Cumulative Probability', fontsize=11)
axes[1, 1].set_title('Cumulative Distribution Function', fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n" + "="*70)
print("MONTE CARLO UNCERTAINTY QUANTIFICATION RESULTS")
print("="*70)
print(f"\nNumber of samples: {stats['n_valid']}")
print(f"Failed simulations: {stats['n_failed']}")
print("\nSwelling Statistics:")
print(f"  Mean:              {stats['mean']:.4f}%")
print(f"  Std deviation:     {stats['std']:.4f}%")
print(f"  Median:            {stats['median']:.4f}%")
print(f"  Range:             [{stats['min']:.4f}, {stats['max']:.4f}]%")
print(f"  90% CI:            [{stats['q5']:.4f}, {stats['q95']:.4f}]%")
print(f"  Coeff. of variation: {stats['std']/stats['mean']*100:.2f}%")

print("\nParameter Correlations with Swelling:")
sorted_correlations = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
for i, (param, corr) in enumerate(sorted_correlations, 1):
    print(f"  {i}. {param:20s}: |r| = {corr:.3f}")

### Interpreting Uncertainty Quantification Results

**Key insights from UQ:**

1. **Mean vs Median**: 
   - Difference indicates distribution skew
   - Similar values → approximately symmetric distribution

2. **Confidence intervals**:
   - 90% CI: Range containing 90% of predictions
   - Wide CI → high uncertainty in predictions
   - Narrow CI → confident predictions

3. **Coefficient of variation** (σ/μ):
   - < 10%: Low relative uncertainty
   - 10-30%: Moderate uncertainty
   - > 30%: High uncertainty

4. **Parameter correlations**:
   - High |r|: Parameter strongly influences swelling
   - Low |r|: Parameter has minimal effect
   - Helps prioritize measurement efforts

## 8. Practical Application: Parameter Prioritization

Based on our sensitivity analysis, we can prioritize parameters for experimental validation and model calibration.

In [ ]:
# Synthesize results from all methods
print("\n" + "="*90)
print("PARAMETER PRIORITIZATION FOR EXPERIMENTAL VALIDATION")
print("="*90)

# Collect OAT elasticities
oat_priorities = {r['parameter']: r['abs_elasticity'] for r in ranking_data}

# Collect Morris mu*
morris_priorities = dict(zip(morris_results.parameter_names, morris_results.mu_star))

# Collect Sobol ST
sobol_priorities = dict(zip(sobol_results.parameter_names, sobol_results.ST[:, 0]))

# Collect correlations from UQ
uq_priorities = {k: v for k, v in sorted_correlations}

# Create priority table
all_params = set(oat_priorities.keys()) | set(morris_priorities.keys()) | \
              set(sobol_priorities.keys()) | set(uq_priorities.keys())

print("\n{:25s} {:>12s} {:>12s} {:>12s} {:>12s} {:>12s}".format(
    "Parameter", "OAT", "Morris", "Sobol", "UQ Corr", "Priority"
))
print("-"*90)

priorities = []
for param in sorted(all_params):
    oat = oat_priorities.get(param, np.nan)
    morris = morris_priorities.get(param, np.nan)
    sobol = sobol_priorities.get(param, np.nan)
    uq = uq_priorities.get(param, np.nan)
    
    # Calculate average normalized priority
    values = [v for v in [oat, morris, sobol, uq] if not np.isnan(v)]
    if values:
        avg_priority = np.mean(values)
        priorities.append((param, avg_priority, oat, morris, sobol, uq))

# Sort by average priority
priorities.sort(key=lambda x: x[1], reverse=True)

for i, (param, avg, oat, morris, sobol, uq) in enumerate(priorities, 1):
    print("{:25s} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f} {:>12d}".format(
        param[:25], 
        oat if not np.isnan(oat) else 0,
        morris if not np.isnan(morris) else 0,
        sobol if not np.isnan(sobol) else 0,
        uq if not np.isnan(uq) else 0,
        i
    ))

print("\n" + "="*90)
print("RECOMMENDATIONS")
print("="*90)

top_3 = priorities[:3]
print("\n🔴 HIGH PRIORITY (measure accurately):")
for param, avg, _, _, _, _ in top_3:
    print(f"   • {param}: Strong influence on predictions")

bottom_3 = priorities[-3:]
print("\n🟢 LOW PRIORITY (can use literature values):")
for param, avg, _, _, _, _ in bottom_3:
    print(f"   • {param}: Minimal influence on predictions")

print("\n📋 VALIDATION STRATEGY:")
print("   1. Focus experimental efforts on high-priority parameters")
print("   2. Use literature values for low-priority parameters")
print("   3. Perform sensitivity analysis whenever using new materials")
print("   4. Update parameter ranges as new data becomes available")

## 9. Helper Functions for Reusable Analysis

Here are some helper functions you can use in your own research.

In [ ]:
def run_comprehensive_sensitivity(param_ranges, sim_time_days=83.3):
    """
    Run a comprehensive sensitivity analysis combining OAT, Morris, and UQ.
    
    Parameters:
    -----------
    param_ranges : list of ParameterRange
        Parameter ranges to analyze
    sim_time_days : float
        Simulation time in days
    
    Returns:
    --------
    dict : Results from all sensitivity methods
    """
    results = {}
    
    # OAT analysis
    print("Running OAT analysis...")
    oat_analyzer = OATAnalyzer(
        parameter_ranges=param_ranges,
        output_names=['swelling'],
        sim_time=sim_time_days * 24 * 3600,
        t_eval_points=100
    )
    results['oat'] = oat_analyzer.run_oat_analysis(
        percent_variations=[-20, -10, 10, 20],
        verbose=False
    )
    
    # Monte Carlo UQ
    print("Running Monte Carlo UQ...")
    results['uq'] = monte_carlo_uq(
        param_ranges=param_ranges,
        n_samples=200,
        sim_time_days=sim_time_days,
        random_state=42
    )
    
    return results


def create_sensitivity_report(results, parameter_name):
    """
    Create a formatted report for a single parameter's sensitivity.
    
    Parameters:
    -----------
    results : dict
        Results from comprehensive_sensitivity()
    parameter_name : str
        Name of parameter to report
    """
    print(f"\n{'='*60}")
    print(f"SENSITIVITY REPORT: {parameter_name}")
    print(f"{'='*60}")
    
    # OAT results
    oat_result = [r for r in results['oat'] if r.parameter_name == parameter_name]
    if oat_result:
        r = oat_result[0]
        sens = r.sensitivities['swelling']
        print(f"\nOAT Analysis:")
        print(f"  Elasticity:     {sens['elasticity']:.3f}")
        print(f"  Normalized:     {sens['normalized']:.3f} ± {sens['std']:.3f}")
        print(f"  Baseline:       {r.baseline_outputs['swelling']:.4f}%")
    
    # UQ correlation
    if parameter_name in results['uq']['parameter_samples']:
        param_samples = results['uq']['parameter_samples'][parameter_name]
        swelling = results['uq']['swelling_samples']
        valid = ~np.isnan(swelling)
        corr = np.corrcoef(param_samples[valid], swelling[valid])[0, 1]
        print(f"\nUQ Correlation:")
        print(f"  Correlation:    {corr:.3f}")
    
    # Interpretation
    if oat_result:
        abs_elast = abs(sens['elasticity'])
        print(f"\nInterpretation:")
        if abs_elast > 1.0:
            print("  ⚠️  HIGH SENSITIVITY - Measure accurately!")
        elif abs_elast > 0.1:
            print("  ⚡ MODERATE SENSITIVITY - Important parameter")
        else:
            print("  ✅ LOW SENSITIVITY - Can use literature values")


def plot_sensitivity_comparison(results, n_top=5):
    """
    Create a comparison plot of parameter sensitivities.
    
    Parameters:
    -----------
    results : dict
        Results from comprehensive_sensitivity()
    n_top : int
        Number of top parameters to show
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Extract OAT elasticities
    params = [r.parameter_name for r in results['oat']]
    elasticities = [abs(r.sensitivities['swelling']['elasticity']) 
                   for r in results['oat']]
    
    # Sort and take top N
    sorted_indices = sorted(range(len(elasticities)), 
                           key=lambda i: elasticities[i], reverse=True)[:n_top]
    
    top_params = [params[i] for i in sorted_indices]
    top_elasticities = [elasticities[i] for i in sorted_indices]
    
    # Create bar plot
    y_pos = np.arange(len(top_params))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_params)))
    ax.barh(y_pos, top_elasticities, color=colors, alpha=0.8, edgecolor='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_params, fontsize=10)
    ax.set_xlabel('|Elasticity|', fontsize=11)
    ax.set_title(f'Top {n_top} Parameters by Sensitivity', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (param, elast) in enumerate(zip(top_params, top_elasticities)):
        ax.text(elast + 0.02, i, f'{elast:.2f}', 
                va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()


print("Helper functions defined:")
print("  • run_comprehensive_sensitivity() - Run OAT + UQ analysis")
print("  • create_sensitivity_report() - Generate parameter report")
print("  • plot_sensitivity_comparison() - Visualize top parameters")

## 10. Summary and Best Practices

### Key Takeaways

**1. Sensitivity Analysis Methods:**
- **OAT**: Fast, simple, local sensitivity
- **Morris**: Global screening, captures nonlinearities
- **Sobol**: Comprehensive variance decomposition
- **Monte Carlo**: Uncertainty quantification

**2. When to Use Each Method:**
- Initial screening → OAT
- Parameter ranking → Morris
- Publishing results → Sobol
- Confidence intervals → Monte Carlo

**3. Interpretation Guidelines:**
- |Elasticity| > 1.0: High sensitivity
- μ* > 0.1: Important parameter
- ST > 0.2: Significant influence on variance
- Coeff. of variation < 10%: Low uncertainty

### Best Practices

1. **Always start with sensitivity analysis**
   - Identify important parameters before detailed studies
   - Save computational resources by fixing insensitive parameters

2. **Use multiple methods when possible**
   - Compare results across methods for robustness
   - OAT for screening, Sobol for final validation

3. **Consider parameter ranges carefully**
   - Based on physical constraints and literature
   - Update ranges as new data becomes available

4. **Document your analysis**
   - Report which method was used and why
   - Include parameter ranges and assumptions
   - Discuss implications of findings

5. **Validate with experiments**
   - Focus measurements on high-priority parameters
   - Use sensitivity results to guide experimental design

### Further Reading

- Saltelli et al. (2008) "Global Sensitivity Analysis: The Primer"
- Iooss & Lemaître (2015) "A Review on Global Sensitivity Analysis"
- `examples/sensitivity_oat_example.py`
- `examples/sensitivity_morris_example.py`
- `examples/sensitivity_sobol_example.py`

### Next Steps

1. **Apply to your research:**
   - Define relevant parameter ranges for your material system
   - Run sensitivity analysis for your specific conditions
   - Use results to guide experimental validation

2. **Explore other notebooks:**
   - `02_Parameter_Sweep_Studies.ipynb`: Detailed parameter exploration
   - `05_Custom_Material_Composition.ipynb`: Composition effects

3. **Advanced topics:**
   - Time-dependent sensitivity (how sensitivities evolve)
   - Regional sensitivity analysis
   - Multi-objective sensitivity (swelling, gas release, etc.)

---

**🎉 Congratulations!** You've completed the Advanced Analysis Techniques notebook.

You now have the skills to:
- Perform sensitivity analysis using multiple methods
- Quantify prediction uncertainties
- Prioritize parameters for experimental validation
- Interpret sensitivity metrics correctly
- Apply these techniques to your research

Happy analyzing! 🚀